# 🤖 Intelligent User Feedback Analysis and Action System
### Multi-Agent AI System using Google Gemini API

**Pipeline Agents:**
1. **CSVReaderAgent** — Reads `app_store_reviews.csv` and `support_emails.csv`
2. **FeedbackClassifierAgent** — Classifies into Bug / Feature Request / Praise / Complaint / Spam
3. **BugAnalysisAgent** — Extracts device, OS, steps-to-reproduce, severity
4. **FeatureExtractorAgent** — Identifies feature details and estimates user impact
5. **TicketCreatorAgent** — Generates structured, formatted tickets
6. **QualityCriticAgent** — Reviews tickets and assigns quality scores

**Outputs:** `generated_tickets.csv`, `processing_log.csv`, `metrics.csv`

## Step 1: Install Dependencies

In [ ]:
!pip install -q google-generativeai pandas
print('✅ Dependencies installed')

## Step 2: Configuration — Paste Your API Key Here

In [ ]:
import os, json, time, re
import pandas as pd
import google.generativeai as genai
from datetime import datetime
from typing import Dict, List

# ════════════════════════════════════════════════════
# ▶  PASTE YOUR GEMINI API KEY BELOW
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')
# ════════════════════════════════════════════════════

MODEL_NAME           = 'gemini-1.5-flash'  # fast & cost-effective
API_DELAY_SEC        = 2                   # seconds between API calls
CONFIDENCE_THRESHOLD = 0.7

PRIORITY_RULES = {
    'Bug':             {'default': 'High',
                        'keywords_critical': ['crash','data loss','deleted','all data','broken','startup','lost']},
    'Feature Request': {'default': 'Medium'},
    'Complaint':       {'default': 'Medium',
                        'keywords_high': ['billing','charge','refund','double charged']},
    'Praise':          {'default': 'Low'},
    'Spam':            {'default': 'Low'},
}

TICKETS_OUTPUT = 'generated_tickets.csv'
LOG_OUTPUT     = 'processing_log.csv'
METRICS_OUTPUT = 'metrics.csv'

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel(MODEL_NAME)
print(f'✅ Gemini configured: {MODEL_NAME}')

## Step 3: Upload CSV Files
Upload the three CSV files: `app_store_reviews.csv`, `support_emails.csv`, `expected_classifications.csv`

In [ ]:
from google.colab import files
print('📂 Upload your 3 CSV files when prompted:')
print('   • app_store_reviews.csv')
print('   • support_emails.csv')
print('   • expected_classifications.csv')
uploaded = files.upload()
print(f'\n✅ Files uploaded: {list(uploaded.keys())}')

## Step 4: Agent Definitions

In [ ]:
# ─── Utility helpers ───────────────────────────────────────────────────────

def call_gemini(prompt: str, max_retries: int = 3) -> str:
    """Call Gemini API with exponential-backoff retry."""
    for attempt in range(max_retries):
        try:
            time.sleep(API_DELAY_SEC)
            resp = model.generate_content(prompt)
            return resp.text.strip()
        except Exception as exc:
            wait = 5 * (attempt + 1)
            print(f'  ⚠️  API error (attempt {attempt+1}/{max_retries}): {exc} — retrying in {wait}s')
            time.sleep(wait)
    return ''


def parse_json_response(text: str) -> dict:
    """Robustly extract a JSON object from a model response."""
    # Strip markdown code fences
    text = re.sub(r'```(?:json)?', '', text).strip()
    # Direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # Find first {...} block
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return {}


def safe_get(row, col: str, default=''):
    """Safely get a column value from a pandas Series row."""
    try:
        val = row[col]
        # Treat NaN / None as missing
        if val is None or (isinstance(val, float) and val != val):
            return default
        return val
    except (KeyError, TypeError):
        return default


print('✅ Utility helpers defined.')

In [ ]:
# ─── AGENT 1: CSV Reader ───────────────────────────────────────────────────

class CSVReaderAgent:
    """Reads and normalises feedback from CSV files into a unified item list."""

    def run(self, reviews_path: str, emails_path: str) -> List[Dict]:
        print('\n📖 [CSVReaderAgent] Reading input files...')
        items = []

        # ── App-store reviews ──
        df_r = pd.read_csv(reviews_path)
        for _, row in df_r.iterrows():
            raw_rating = safe_get(row, 'rating', None)
            try:
                rating = int(float(raw_rating)) if raw_rating is not None else None
            except (ValueError, TypeError):
                rating = None
            items.append({
                'source_id':   str(safe_get(row, 'review_id', f'R{_}')),
                'source_type': 'app_store_review',
                'platform':    str(safe_get(row, 'platform', '')),
                'rating':      rating,
                'text':        str(safe_get(row, 'review_text', '')),
                'author':      str(safe_get(row, 'user_name', '')),
                'date':        str(safe_get(row, 'date', '')),
                'app_version': str(safe_get(row, 'app_version', '')),
                'subject':     '',
                'sender':      '',
            })

        # ── Support emails ──
        df_e = pd.read_csv(emails_path)
        for _, row in df_e.iterrows():
            items.append({
                'source_id':   str(safe_get(row, 'email_id', f'E{_}')),
                'source_type': 'support_email',
                'platform':    '',
                'rating':      None,
                'text':        str(safe_get(row, 'body', '')),
                'author':      str(safe_get(row, 'sender_email', '')),
                'date':        str(safe_get(row, 'timestamp', '')),
                'app_version': '',
                'subject':     str(safe_get(row, 'subject', '')),
                'sender':      str(safe_get(row, 'sender_email', '')),
            })

        print(f'  ✅ Loaded {len(df_r)} reviews + {len(df_e)} emails = {len(items)} total items')
        return items


# ─── AGENT 2: Feedback Classifier ──────────────────────────────────────────

class FeedbackClassifierAgent:
    """Categorises each feedback item using Gemini."""

    _PROMPT = (
        'You are a feedback classification expert for a mobile productivity app.\n\n'
        'Classify the following feedback into EXACTLY ONE category:\n'
        '- Bug: technical problem, crash, error, data loss, sync failure\n'
        '- Feature Request: new functionality, improvement suggestion\n'
        '- Praise: positive feedback, compliment, satisfaction\n'
        '- Complaint: dissatisfaction about pricing, support, policies (NOT technical bugs)\n'
        '- Spam: irrelevant, promotional, incoherent\n\n'
        'Source Type: {source_type}\n'
        'Subject: {subject}\n'
        'Rating: {rating}\n'
        'Text: {text}\n\n'
        'Respond ONLY with valid JSON, no markdown, no extra text:\n'
        '{{"category": "<Bug|Feature Request|Praise|Complaint|Spam>", '
        '"confidence": <0.0-1.0>, '
        '"reasoning": "<one sentence>"}}'
    )

    def classify(self, item: Dict) -> Dict:
        prompt = self._PROMPT.format(
            source_type = item['source_type'],
            subject     = item.get('subject') or 'N/A',
            rating      = item.get('rating') or 'N/A',
            text        = item['text'][:800],
        )
        result     = parse_json_response(call_gemini(prompt))
        category   = result.get('category', 'Complaint')
        confidence = float(result.get('confidence', 0.5))
        reasoning  = str(result.get('reasoning', ''))

        # Sanity-check: valid category names only
        valid = {'Bug', 'Feature Request', 'Praise', 'Complaint', 'Spam'}
        if category not in valid:
            category = 'Complaint'

        # Low-confidence fallback using star rating
        if confidence < CONFIDENCE_THRESHOLD and item.get('rating') is not None:
            r = item['rating']
            if isinstance(r, (int, float)) and not (isinstance(r, float) and r != r):
                if r <= 2 and category not in ('Spam',):
                    category = 'Bug'
                elif r >= 4 and category not in ('Spam', 'Feature Request'):
                    category = 'Praise'

        return {'category': category, 'confidence': confidence, 'reasoning': reasoning}


# ─── AGENT 3: Bug Analysis ──────────────────────────────────────────────────

class BugAnalysisAgent:
    """Extracts structured technical details from bug reports."""

    _PROMPT = (
        'You are a technical support engineer analysing a bug report for a mobile productivity app.\n\n'
        'Bug Report Text:\n{text}\n\n'
        'Extract all available technical details. If a field is not mentioned, use "Unknown" or "Not provided".\n'
        'Respond ONLY with valid JSON, no markdown:\n'
        '{{"device": "<model or Unknown>", '
        '"os": "<OS + version or Unknown>", '
        '"app_version": "<version or Unknown>", '
        '"steps_to_reproduce": "<numbered steps or Not provided>", '
        '"severity": "<Critical|High|Medium|Low>", '
        '"severity_reason": "<brief reason>", '
        '"affected_feature": "<feature name>", '
        '"workaround_available": "<Yes|No|Unknown>"}}'
    )

    def analyse(self, item: Dict) -> Dict:
        result = parse_json_response(call_gemini(
            self._PROMPT.format(text=item['text'][:1000])
        ))
        if not result:
            result = {
                'device': item.get('platform', 'Unknown'),
                'os': 'Unknown', 'app_version': item.get('app_version', 'Unknown'),
                'steps_to_reproduce': 'Not provided', 'severity': 'High',
                'severity_reason': 'Bug reported by user',
                'affected_feature': 'Unknown', 'workaround_available': 'Unknown',
            }
        return result


# ─── AGENT 4: Feature Extractor ─────────────────────────────────────────────

class FeatureExtractorAgent:
    """Extracts structured feature-request details and estimates impact."""

    _PROMPT = (
        'You are a product manager analysing a feature request for a mobile productivity app.\n\n'
        'Feature Request:\n{text}\n\n'
        'Respond ONLY with valid JSON, no markdown:\n'
        '{{"feature_name": "<short name>", '
        '"description": "<what the user wants>", '
        '"user_impact": "<High|Medium|Low>", '
        '"estimated_demand": "<High|Medium|Low>", '
        '"implementation_complexity": "<High|Medium|Low>", '
        '"rationale": "<one sentence on why this matters>"}}'
    )

    def extract(self, item: Dict) -> Dict:
        result = parse_json_response(call_gemini(
            self._PROMPT.format(text=item['text'][:800])
        ))
        if not result:
            result = {
                'feature_name': 'Unspecified Feature',
                'description': item['text'][:200],
                'user_impact': 'Medium', 'estimated_demand': 'Medium',
                'implementation_complexity': 'Medium', 'rationale': '',
            }
        return result


# ─── AGENT 5: Ticket Creator ─────────────────────────────────────────────────

class TicketCreatorAgent:
    """Generates a structured, formatted engineering ticket."""

    _TAG_MAP = {'Bug': 'BUG', 'Feature Request': 'FEATURE',
                'Praise': 'PRAISE', 'Complaint': 'COMPLAINT', 'Spam': 'SPAM'}

    _PROMPT = (
        'You are a product engineer creating an engineering ticket from user feedback.\n\n'
        'Category: {category}\n'
        'Source: {source_type} | ID: {source_id}\n'
        'Feedback: {text}\n'
        'Technical Info: {technical_info}\n\n'
        'Create a well-structured ticket. The title MUST start with [{tag}].\n'
        'Respond ONLY with valid JSON, no markdown:\n'
        '{{"title": "[{tag}] <clear actionable title>", '
        '"description": "<2-3 sentences describing the issue or request>", '
        '"acceptance_criteria": "<what done looks like>", '
        '"labels": ["<label1>", "<label2>"], '
        '"component": "<affected component>", '
        '"suggested_assignee_team": "<Engineering|Product|Support|QA>", '
        '"estimated_effort": "<XS|S|M|L|XL>", '
        '"user_story": "As a user, I want... so that..."}}'
    )

    def _determine_priority(self, category: str, text: str, tech_info: Dict) -> str:
        rule     = PRIORITY_RULES.get(category, {'default': 'Medium'})
        priority = rule['default']
        lower    = text.lower()
        if category == 'Bug':
            if any(k in lower for k in rule.get('keywords_critical', [])):
                priority = 'Critical'
            elif tech_info.get('severity') in ('Critical',):
                priority = 'Critical'
            elif tech_info.get('severity') == 'High':
                priority = 'High'
        elif category == 'Complaint':
            if any(k in lower for k in rule.get('keywords_high', [])):
                priority = 'High'
        return priority

    def create(self, item: Dict, classification: Dict, technical_info: Dict) -> Dict:
        category = classification['category']
        priority = self._determine_priority(category, item['text'], technical_info)
        tag      = self._TAG_MAP.get(category, 'MISC')

        tech_str = json.dumps(technical_info) if technical_info else 'N/A'
        prompt   = self._PROMPT.format(
            category      = category,
            tag           = tag,
            source_type   = item['source_type'],
            source_id     = item['source_id'],
            text          = item['text'][:600],
            technical_info= tech_str[:400],
        )
        td = parse_json_response(call_gemini(prompt))

        # Fallback title if model returned nothing useful
        if not td.get('title'):
            fallback = item.get('subject') or item['text'][:60]
            td['title'] = f'[{tag}] {fallback}'

        # Ensure labels is a list of strings
        raw_labels = td.get('labels', [category])
        if not isinstance(raw_labels, list):
            raw_labels = [str(raw_labels)]
        labels_str = ', '.join(str(l) for l in raw_labels)

        return {
            'ticket_id':            f'TKT-{item["source_id"]}',
            'source_id':            item['source_id'],
            'source_type':          item['source_type'],
            'category':             category,
            'priority':             priority,
            'title':                td.get('title', ''),
            'description':          td.get('description', ''),
            'acceptance_criteria':  td.get('acceptance_criteria', ''),
            'component':            td.get('component', 'General'),
            'labels':               labels_str,
            'suggested_assignee':   td.get('suggested_assignee_team', ''),
            'estimated_effort':     td.get('estimated_effort', ''),
            'user_story':           td.get('user_story', ''),
            'technical_details':    tech_str if technical_info else '',
            'original_text':        item['text'][:300],
            'confidence':           round(classification['confidence'], 3),
            'classification_note':  classification.get('reasoning', ''),
            'created_at':           datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'status':               'Open',
        }


# ─── AGENT 6: Quality Critic ─────────────────────────────────────────────────

class QualityCriticAgent:
    """Reviews generated tickets for completeness, clarity, and accuracy."""

    _PROMPT = (
        'You are a QA engineer reviewing an auto-generated engineering ticket.\n\n'
        'Ticket Title: {title}\n'
        'Category: {category}\n'
        'Priority: {priority}\n'
        'Description: {description}\n'
        'Acceptance Criteria: {acceptance_criteria}\n'
        'Original Feedback: {original_text}\n\n'
        'Score this ticket from 1-10 and flag any issues.\n'
        'Respond ONLY with valid JSON, no markdown:\n'
        '{{"quality_score": <1-10>, '
        '"title_clear": <true|false>, '
        '"priority_appropriate": <true|false>, '
        '"description_complete": <true|false>, '
        '"issues": ["<issue or empty list>"], '
        '"suggested_improvement": "<one suggestion or empty string>", '
        '"approved": <true|false>}}'
    )

    def review(self, ticket: Dict) -> Dict:
        prompt = self._PROMPT.format(
            title                = ticket.get('title', ''),
            category             = ticket.get('category', ''),
            priority             = ticket.get('priority', ''),
            description          = ticket.get('description', ''),
            acceptance_criteria  = ticket.get('acceptance_criteria', ''),
            original_text        = ticket.get('original_text', ''),
        )
        result = parse_json_response(call_gemini(prompt))
        if not result:
            return {'quality_score': 7, 'approved': True, 'issues': [],
                    'suggested_improvement': '', 'title_clear': True,
                    'priority_appropriate': True, 'description_complete': True}
        # Ensure issues is always a list of strings
        issues = result.get('issues', [])
        if not isinstance(issues, list):
            issues = [str(issues)]
        result['issues'] = [str(i) for i in issues]
        return result


print('✅ All 6 agent classes defined.')

## Step 5: Pipeline Orchestrator

In [ ]:
class FeedbackPipeline:
    """Orchestrates all 6 agents and persists results to CSV."""

    def __init__(self):
        self.reader     = CSVReaderAgent()
        self.classifier = FeedbackClassifierAgent()
        self.bug_agent  = BugAnalysisAgent()
        self.feat_agent = FeatureExtractorAgent()
        self.creator    = TicketCreatorAgent()
        self.critic     = QualityCriticAgent()
        self.tickets    : List[Dict] = []
        self.logs       : List[Dict] = []
        self.metrics    = {
            'total': 0, 'by_category': {}, 'by_priority': {},
            'approved': 0, 'avg_quality': 0.0,
        }

    # ── internal logger ──
    def _log(self, source_id: str, stage: str, status: str, detail: str = '') -> None:
        self.logs.append({
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'source_id': source_id,
            'stage':     stage,
            'status':    status,
            'detail':    detail,
        })

    # ── main run ──
    def run(self, reviews_path: str, emails_path: str) -> List[Dict]:
        items = self.reader.run(reviews_path, emails_path)
        self.metrics['total'] = len(items)
        quality_scores: List[float] = []

        for idx, item in enumerate(items):
            sid = item['source_id']
            print(f'\n[{idx+1}/{len(items)}] {sid} ({item["source_type"]})')

            # 1. Classify
            try:
                clf = self.classifier.classify(item)
                self._log(sid, 'Classification', 'OK',
                          f"{clf['category']} ({clf['confidence']:.2f})")
                print(f'  📌 {clf["category"]} | confidence={clf["confidence"]:.2f}')
            except Exception as exc:
                clf = {'category': 'Complaint', 'confidence': 0.5, 'reasoning': str(exc)}
                self._log(sid, 'Classification', 'ERROR', str(exc))
                print(f'  ❌ Classification failed: {exc}')

            # 2. Deep analysis (Bug or Feature only)
            tech_info: Dict = {}
            if clf['category'] == 'Bug':
                try:
                    tech_info = self.bug_agent.analyse(item)
                    self._log(sid, 'BugAnalysis', 'OK', f"severity={tech_info.get('severity','?')}")
                    print(f'  🐛 severity={tech_info.get("severity","?")} | device={tech_info.get("device","?")} | os={tech_info.get("os","?")}')  # noqa
                except Exception as exc:
                    self._log(sid, 'BugAnalysis', 'ERROR', str(exc))

            elif clf['category'] == 'Feature Request':
                try:
                    tech_info = self.feat_agent.extract(item)
                    self._log(sid, 'FeatureExtract', 'OK', f"impact={tech_info.get('user_impact','?')}")
                    print(f'  ✨ feature={tech_info.get("feature_name","?")} | impact={tech_info.get("user_impact","?")}')  # noqa
                except Exception as exc:
                    self._log(sid, 'FeatureExtract', 'ERROR', str(exc))

            # 3. Create ticket
            try:
                ticket = self.creator.create(item, clf, tech_info)
                self._log(sid, 'TicketCreation', 'OK', ticket['title'][:80])
                print(f'  🎫 {ticket["title"][:75]}')
            except Exception as exc:
                self._log(sid, 'TicketCreation', 'ERROR', str(exc))
                print(f'  ❌ Ticket creation failed: {exc}')
                continue

            # 4. Quality review
            try:
                review = self.critic.review(ticket)
                ticket['quality_score'] = int(review.get('quality_score', 7))
                ticket['qa_approved']   = bool(review.get('approved', True))
                ticket['qa_issues']     = '; '.join(review.get('issues', []))
                ticket['qa_suggestion'] = str(review.get('suggested_improvement', ''))
                quality_scores.append(ticket['quality_score'])
                if ticket['qa_approved']:
                    self.metrics['approved'] += 1
                self._log(sid, 'QualityReview', 'OK',
                          f"score={ticket['quality_score']} approved={ticket['qa_approved']}")
                print(f'  ✅ quality={ticket["quality_score"]}/10 | approved={ticket["qa_approved"]}')
            except Exception as exc:
                ticket['quality_score'] = 7
                ticket['qa_approved']   = True
                ticket['qa_issues']     = ''
                ticket['qa_suggestion'] = ''
                self._log(sid, 'QualityReview', 'ERROR', str(exc))

            self.tickets.append(ticket)

            # Update aggregate metrics
            cat = clf['category']
            pri = ticket['priority']
            self.metrics['by_category'][cat] = self.metrics['by_category'].get(cat, 0) + 1
            self.metrics['by_priority'][pri]  = self.metrics['by_priority'].get(pri, 0) + 1

        self.metrics['avg_quality'] = (
            round(sum(quality_scores) / len(quality_scores), 2) if quality_scores else 0.0
        )

        print(f'\n{"="*60}')
        print(f'✅ Pipeline complete! {len(self.tickets)} tickets generated.')
        print(f'   Avg quality score : {self.metrics["avg_quality"]}/10')
        print(f'   QA approved       : {self.metrics["approved"]}/{len(self.tickets)}')
        print(f'   Categories        : {self.metrics["by_category"]}')
        print(f'   Priorities        : {self.metrics["by_priority"]}')
        return self.tickets

    # ── save all outputs ──
    def save_outputs(self) -> None:
        if not self.tickets:
            print('⚠️  No tickets to save.')
            return

        pd.DataFrame(self.tickets).to_csv(TICKETS_OUTPUT, index=False)
        print(f'💾 {TICKETS_OUTPUT} — {len(self.tickets)} rows')

        pd.DataFrame(self.logs).to_csv(LOG_OUTPUT, index=False)
        print(f'💾 {LOG_OUTPUT} — {len(self.logs)} rows')

        rows = [
            {'metric': 'total_items_processed', 'value': self.metrics['total']},
            {'metric': 'total_tickets_created', 'value': len(self.tickets)},
            {'metric': 'qa_approved_tickets',   'value': self.metrics['approved']},
            {'metric': 'avg_quality_score',     'value': self.metrics['avg_quality']},
        ]
        for cat, cnt in self.metrics['by_category'].items():
            rows.append({'metric': f'category_{cat.lower().replace(" ","_")}', 'value': cnt})
        for pri, cnt in self.metrics['by_priority'].items():
            rows.append({'metric': f'priority_{pri.lower()}', 'value': cnt})
        pd.DataFrame(rows).to_csv(METRICS_OUTPUT, index=False)
        print(f'💾 {METRICS_OUTPUT} — {len(rows)} rows')


print('✅ FeedbackPipeline orchestrator defined.')

## Step 6: Run the Full Pipeline

In [ ]:
pipeline = FeedbackPipeline()
tickets  = pipeline.run('app_store_reviews.csv', 'support_emails.csv')
pipeline.save_outputs()

## Step 7: Accuracy Evaluation vs Expected Classifications

In [ ]:
expected_df  = pd.read_csv('expected_classifications.csv')
generated_df = pd.read_csv(TICKETS_OUTPUT)

merged = generated_df.merge(
    expected_df[['source_id', 'category', 'priority']].rename(
        columns={'category': 'expected_category', 'priority': 'expected_priority'}
    ),
    on='source_id', how='inner'
)

total = len(merged)
if total == 0:
    print('⚠️  No source_id matches found between generated tickets and expected classifications.')
else:
    cat_correct = (merged['category'] == merged['expected_category']).sum()
    pri_correct = (merged['priority']  == merged['expected_priority']).sum()

    print(f'\n📊 Accuracy Report ({total} matched items)')
    print(f'  Category Accuracy : {cat_correct}/{total} = {100*cat_correct/total:.1f}%')
    print(f'  Priority Accuracy : {pri_correct}/{total} = {100*pri_correct/total:.1f}%')

    mismatches = merged[merged['category'] != merged['expected_category']]
    if not mismatches.empty:
        print(f'\n⚠️  Category Mismatches ({len(mismatches)}):')
        for _, row in mismatches.iterrows():
            print(f'    {row["source_id"]}: got={row["category"]}  expected={row["expected_category"]}')
    else:
        print('\n🎉 All categories matched expected classifications!')

    display(merged[['source_id','category','expected_category','priority','expected_priority']].head(20))

## Step 8: Inspect Generated Tickets

In [ ]:
df = pd.read_csv(TICKETS_OUTPUT)
print(f'Total tickets: {len(df)}')
print(f'\nCategory distribution:')
print(df['category'].value_counts().to_string())
print(f'\nPriority distribution:')
print(df['priority'].value_counts().to_string())
print(f'\nAvg quality score: {df["quality_score"].mean():.2f}/10')
print(f'QA approval rate : {df["qa_approved"].mean()*100:.1f}%')
print(f'\n--- First 10 tickets ---')
display(df[['ticket_id','category','priority','title','quality_score','qa_approved']].head(10))

## Step 9: Manual Override
Use this cell to manually edit any ticket before final export.

In [ ]:
# ── Manual Override Utility ────────────────────────────────────────────────
# Change the values below and run this cell to update the ticket.

OVERRIDE_TICKET_ID = 'TKT-R001'  # Change to the ticket you want to edit

OVERRIDES = {
    # 'priority':    'Critical',
    # 'category':    'Bug',
    # 'title':       '[BUG] My corrected title',
    # 'description': 'Updated description here.',
    # 'status':      'In Progress',
}

if OVERRIDES:
    df = pd.read_csv(TICKETS_OUTPUT)
    mask = df['ticket_id'] == OVERRIDE_TICKET_ID
    if mask.any():
        for col, val in OVERRIDES.items():
            if col in df.columns:
                df.loc[mask, col] = val
                print(f'✏️  Updated {OVERRIDE_TICKET_ID}.{col} → {val}')
            else:
                print(f'⚠️  Column "{col}" not found in tickets — skipped')
        df.to_csv(TICKETS_OUTPUT, index=False)
        print(f'\n💾 Saved changes to {TICKETS_OUTPUT}')
        display(df[df['ticket_id'] == OVERRIDE_TICKET_ID])
    else:
        print(f'⚠️  Ticket ID "{OVERRIDE_TICKET_ID}" not found in {TICKETS_OUTPUT}')
else:
    print('ℹ️  No overrides specified. Populate the OVERRIDES dict and re-run.')

## Step 10: Download Output Files

In [ ]:
from google.colab import files
import os

for fpath in [TICKETS_OUTPUT, LOG_OUTPUT, METRICS_OUTPUT]:
    if os.path.exists(fpath):
        files.download(fpath)
        print(f'⬇️  Downloading {fpath}')
    else:
        print(f'⚠️  {fpath} not found — run the pipeline first.')

print('\n✅ Done. Check your browser downloads.')

## Step 10: Run the Streamlit Interface
Run this cell to launch the UI and click the localtunnel link!

In [ ]:
import os
import urllib
from google.colab import userdata

# Expose the API key to the environment so Streamlit and CrewAI can read it
os.environ['GEMINI_API_KEY'] = userdata.get('GOOGLE_API_KEY')

print("Tunnel Password/Endpoint IP:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
!npm install -g localtunnel
!streamlit run app.py & npx localtunnel --port 8501
